# Speech-To-Text (STT) using Whisper

In [ ]:
!pip install -U openai-whisper

In [ ]:
import whisper

model_stt = whisper.load_model("base") # Renamed to model_stt to avoid conflicts

In [ ]:
def transcribe_audio(audio_path):
    """
    Transcribes the given audio file to text using the loaded Whisper model.
    """
    result = model_stt.transcribe(audio_path)
    return result["text"]

# Example usage:
audio_file = "path_to_your_audio.wav" # You'll need an audio file
transcribed_text = transcribe_audio(audio_file)
print(transcribed_text)

# Text-to-Speech (TTS) using Tacotron2 and Waveglow

In [ ]:
Text-to-Speech (TTS) using Tacotron2 and Waveglow

In [ ]:
import torch

# Load Tacotron2
tacotron2 = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_tacotron2', model_math='fp16')
tacotron2 = tacotron2.to('cuda' if torch.cuda.is_available() else 'cpu')
tacotron2.eval()

# Load Waveglow
waveglow = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_waveglow', model_math='fp16')
waveglow = waveglow.remove_weightnorm(waveglow)
waveglow = waveglow.to('cuda' if torch.cuda.is_available() else 'cpu')
waveglow.eval()

# Load utility for text processing
utils = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_tts_utils')

In [ ]:
from IPython.display import Audio

def text_to_speech(text_input):
    """
    Converts text input to a playable audio signal.
    """
    sequences, lengths = utils.prepare_input_sequence([text_input])
    sequences = sequences.to('cuda' if torch.cuda.is_available() else 'cpu')
    lengths = lengths.to('cuda' if torch.cuda.is_available() else 'cpu')

    with torch.no_grad():
        mel, _, _ = tacotron2.infer(sequences, lengths)
        audio_generated = waveglow.infer(mel)
    
    audio_numpy = audio_generated[0].data.cpu().numpy()
    rate_tts = 22050 # Standard sampling rate for these models
    
    return audio_numpy, rate_tts

# Example usage:
my_text = "Hello, this is a test of text to speech in bart_try notebook."
generated_speech, sample_rate = text_to_speech(my_text)
display(Audio(generated_speech, rate=sample_rate))

# To save the audio:
from scipy.io.wavfile import write
write("generated_audio.wav", sample_rate, generated_speech)